# Titanic: A Complete Preprocessing and Pipeline Workflow

Personal notebook, not part of the course materials. Titanic dataset (real missing values, real categorical columns), raw data to a defined ML problem, EDA, feature engineering, a preprocessing Pipeline, two classifiers, evaluation.

### Table of contents
1. [From raw data to an ML problem](#1.-From-raw-data-to-an-ML-problem)
2. [Exploratory data analysis: know your data](#2.-Exploratory-data-analysis:-know-your-data)
3. [Feature Engineering](#3.-Feature-Engineering)
4. [Building the preprocessing Pipeline](#4.-Building-the-preprocessing-Pipeline)
5. [Baseline model: Logistic Regression](#5.-Baseline-model:-Logistic-Regression)
6. [Evaluate the baseline](#6.-Evaluate-the-baseline)
7. [Try a second model: Random Forest](#7.-Try-a-second-model:-Random-Forest)
8. [Compare the two models](#8.-Compare-the-two-models)
9. [Which features mattered, according to the forest](#9.-Which-features-mattered,-according-to-the-forest)
10. [A peek ahead: Cross Validation on the whole pipeline](#10.-A-peek-ahead:-Cross-Validation-on-the-whole-pipeline)
11. [Final pipeline](#11.-Final-pipeline)

## 1. From raw data to an ML problem

In [2]:
# real Kaggle-style Titanic data: real missing values, real categorical columns, one function call
from sklearn.datasets import fetch_openml

titanic = fetch_openml("titanic", version=1, as_frame=True)
df = titanic.frame
df.head(10)

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
5,1,1,"Anderson, Mr. Harry",male,48.0000,0,0,19952,26.5500,E12,S,3,NaN,"New York, NY"
6,1,1,"Andrews, Miss. Kornelia Theodosia",female,63.0000,1,0,13502,77.9583,D7,S,10,NaN,"Hudson, NY"
7,1,0,"Andrews, Mr. Thomas Jr",male,39.0000,0,0,112050,0.0000,A36,S,NaN,NaN,"Belfast, NI"
8,1,1,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",female,53.0000,2,0,11769,51.4792,C101,S,D,NaN,"Bayside, Queens, NY"
9,1,0,"Artagaveytia, Mr. Ramon",male,71.0000,0,0,PC 17609,49.5042,NaN,C,NaN,22.0,"Montevideo, Uruguay"


`fetch_openml`'s `.DESCR` is just narrative history here, not a per-column table like `load_wine`/`load_breast_cancer`. For reference:

| Column | Meaning |
|---|---|
| `pclass` | Ticket class: 1 = 1st, 2 = 2nd, 3 = 3rd (proxy for socio-economic status) |
| `survived` | Target: 0 = died, 1 = survived |
| `name` | Full name, including title (`Mr.`, `Mrs.`, ...) |
| `sex` | male / female |
| `age` | Age in years, fractional for infants |
| `sibsp` | Number of siblings/spouses aboard |
| `parch` | Number of parents/children aboard |
| `ticket` | Ticket number |
| `fare` | Passenger fare |
| `cabin` | Cabin number |
| `embarked` | Port of embarkation: C = Cherbourg, Q = Queenstown, S = Southampton |
| `boat` | Lifeboat number, survivors only (dropped, leakage) |
| `body` | Body recovery number, non-survivors only (dropped, leakage) |
| `home.dest` | Home / destination |

In [2]:
# boat/body reveal the outcome directly (had a lifeboat number -> survived, body recovered -> did not)
df = df.drop(columns=["boat", "body"])
df.shape

(1309, 12)

In [3]:
X = df.drop(columns="survived")
y = df["survived"].astype(int)
y.value_counts()

survived
0    809
1    500
Name: count, dtype: int64

In [4]:
# split before any exploration, so section 2 below only ever looks at the training set
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((1047, 11), (262, 11))

## 2. Exploratory data analysis: know your data

In [5]:
X_train.dtypes

pclass          int64
name              str
sex          category
age           float64
sibsp           int64
parch           int64
ticket            str
fare          float64
cabin             str
embarked     category
home.dest         str
dtype: object

In [6]:
# multiple columns, multiple missingness levels: everything section 3/4 needs to handle
X_train.isna().mean().sort_values(ascending=False) * 100

cabin        78.510029
home.dest    42.979943
age          19.961796
fare          0.095511
sex           0.000000
pclass        0.000000
name          0.000000
ticket        0.000000
parch         0.000000
sibsp         0.000000
embarked      0.000000
dtype: float64

### Could we just drop the rows?

In [7]:
# dropping any row missing anything is a bloodbath, mostly because of cabin alone
X_train.dropna().shape

(188, 11)

In [8]:
# even setting cabin aside (we'll engineer around it, not drop it), age's 20% still costs half the data
X_train.drop(columns="cabin").dropna().shape

(550, 10)

Neither is acceptable here. Impute and engineer instead (sections 3, 4).

In [9]:
X_train[["sex", "embarked", "pclass"]].describe(include="all")

,sex,embarked,pclass
count,1047,1047,1047.000000
unique,2,3,NaN
top,male,S,NaN
freq,675,730,NaN
mean,NaN,NaN,2.314231
std,NaN,NaN,0.831742
min,NaN,NaN,1.000000
25%,NaN,NaN,2.000000
50%,NaN,NaN,3.000000
75%,NaN,NaN,3.000000


In [10]:
# a handful of very expensive tickets; matters for mean vs median below
X_train["fare"].sort_values(ascending=False).head()

302    512.3292
49     512.3292
112    263.0000
115    263.0000
111    263.0000
Name: fare, dtype: float64

### Mean vs median, on `fare`

In [11]:
mean_fare = X_train["fare"].mean()
median_fare = X_train["fare"].median()
print(f"mean:   {mean_fare:.2f}")
print(f"median: {median_fare:.2f}")

mean:   32.17
median: 13.82


The 512-fare tickets pull the mean to more than double the median. Median is the safer fill for section 4.

## 3. Feature Engineering

### Title, extracted from name

In [12]:
def add_title(df):
    df = df.copy()
    title = df["name"].str.extract(r",\s*([^.]*)\.")[0]
    title = title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    common = ["Mr", "Miss", "Mrs", "Master"]
    df["title"] = title.where(title.isin(common), "Rare")
    return df

X_train = add_title(X_train)
X_test = add_title(X_test)
X_train["title"].value_counts()

title
Mr        606
Miss      214
Mrs       156
Master     51
Rare       20
Name: count, dtype: int64

### Family size and traveling alone

In [13]:
def add_family_features(df):
    df = df.copy()
    df["family_size"] = df["sibsp"] + df["parch"] + 1
    df["is_alone"] = (df["family_size"] == 1).astype(int)
    return df

X_train = add_family_features(X_train)
X_test = add_family_features(X_test)
X_train[["sibsp", "parch", "family_size", "is_alone"]].head()

,sibsp,parch,family_size,is_alone
999,0,0,1,1
392,1,0,2,0
628,4,2,7,0
1165,0,0,1,1
604,0,0,1,1


### Cabin: missingness itself is a signal

In [14]:
def add_cabin_features(df):
    df = df.copy()
    df["has_cabin"] = df["cabin"].notna().astype(int)
    df["deck"] = df["cabin"].str[0].fillna("U")
    return df

X_train = add_cabin_features(X_train)
X_test = add_cabin_features(X_test)
X_train["deck"].value_counts()

deck
U    822
C     73
B     48
D     41
E     31
A     16
F     12
G      3
T      1
Name: count, dtype: int64

### Dropping columns we've now replaced

In [15]:
drop_cols = ["name", "sibsp", "parch", "ticket", "cabin", "home.dest"]
X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)
X_train.columns.tolist()

['pclass',
 'sex',
 'age',
 'fare',
 'embarked',
 'title',
 'family_size',
 'is_alone',
 'has_cabin',
 'deck']